# 02 - Ingeniería de Características

## Descripción General

La ingeniería de características (*feature Engineering*) es el proceso de transformar datos brutos en entradas significativas (características) que ayuden a los modelos de machine learning a aprender patrones de manera más efectiva y mejorar la precisión de las predicciones. Esto implica la creación de nuevas características, transformación de las características existentes, imputación de valores faltantes, codificación de datos categóricos y selección de las características más relevantes.

Dado que el rendimiento de un modelo depende en gran medida de la calidad de los datos de entrenamiento, la ingeniería de características se convierte en un paso crucial del preprocesamiento. El objetivo de esta sección será transformar y seleccionar los aspectos más relevantes de los datos brutos, alineándolos tanto con la tarea predictiva como con las necesidades específicas del modelo, para maximizar su capacidad de aprendizaje y generalización.

En esta etapa dejamos de “explorar” y empezamos a construir lo que el modelo realmente va a usar. Todo el EDA previo tuvo un propósito claro: identificar qué variables contienen señal y cuáles no. Aquí convertiremos ese análisis en un proceso reproducible.

Partiremos del conjunto de datos `df_features` que contiene las características candidatas al modelo y construiremos un conjunto de variables (X) que capturen las señales mas relevantes sobre los factores que influyen en la fijación de precios de los alojamientos de Airbnb en la Ciudad de México. Para lograrlo, se dividirá el proceso en dos niveles:

- **Creación de features (`build_features`):**  
  Generar nuevas variables a partir de los datos originales. Estas pueden incluir features o métricas derivadas, combinaciones de atributos o indicadores que aporten señales adicionales al modelo.

- **Transformación de features (`transform_features`):**  
  Modificar las variables existentes para optimizar su representación. Esto abarca operaciones como normalización, escalado, codificación de categorías o transformaciones matemáticas que mejoren la capacidad predictiva.

**Flujo completo**


```text
df_clean
   ↓
df_features
   ↓
build_features()
   ↓
transform_features()
   ↓
df_model → X listo para modelado

## Importación de Librerías y Carga del Dataset

In [1]:
# Import libraries, modules and dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Load the autoreload extension to automatically reload modules when they change
# Set autoreload mode to 2: reload all modules before executing user code
%load_ext autoreload
%autoreload 2 

# Import validate features module
from src.features.validate_feature import validate_features

# Load dataset
df_features = pd.read_csv("../data/processed/df_features.csv")

# Preview first rows
df_features.head()

,listing_url,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,number_of_reviews,price,clean_price,log_price,price_per_guest
0,https://www.airbnb.com/rooms/35797,"Mexico City, D.f., Mexico",Cuajimalpa de Morelos,NaN,19.38283,-99.27178,Entire villa,Entire home/apt,2,1.0,...,NaN,NaN,NaN,NaN,NaN,0,"$3,673.00",3673.0,8.208764,1836.500000
1,https://www.airbnb.com/rooms/44616,NaN,Cuauhtémoc,NaN,19.41162,-99.17794,Entire home,Entire home/apt,14,5.5,...,4.70,4.87,4.78,4.98,4.47,65,"$18,000.00",18000.0,9.798127,1285.714286
2,https://www.airbnb.com/rooms/56074,"Mexico City, DF, Mexico",Cuauhtémoc,NaN,19.43977,-99.15605,Entire condo,Entire home/apt,2,1.0,...,4.88,4.98,4.94,4.76,4.79,84,$591.00,591.0,6.381816,295.500000
3,https://www.airbnb.com/rooms/165772,"Mexico City, Federal District, Mexico",Miguel Hidalgo,NaN,19.40826,-99.18659,Entire home,Entire home/apt,16,5.0,...,4.84,4.91,4.89,4.75,4.90,386,"$3,673.00",3673.0,8.208764,229.562500
4,https://www.airbnb.com/rooms/171109,"Mexico City, Federal District, Mexico",Benito Juárez,NaN,19.39675,-99.17581,Private room in rental unit,Private room,2,1.0,...,4.61,4.98,4.95,4.97,4.83,123,$321.00,321.0,5.771441,160.500000


## Creación de Características (Build Features)

En esta sección se construirán las variables derivadas que constituirán el conjunto de entrada (X) para el modelo. A partir de las variables originales, se generaran nuevas features con capacidad explicativa. El enfoque seguido no es arbitrario: cada feature propuesta proviene del análisis exploratorio previo, donde se identificaron patrones, relaciones y posibles fuentes de información predictiva.

Durante este proceso:

- Se diseñrann variables con capacidad explicativa (por ejemplo, ubicación, amenidades, comportamiento del host).
- Se combinan y transformarán variables existentes para capturar relaciones no evidentes.
- Se validará cada feature de manera individual para evaluar su distribución, coherencia y posible relación con la variable objetivo.

El objetivo no es aumentar el número de variables, sino capturar mejor la señal relevante: ubicación, características del listing, calidad percibida y contexto del entorno. Estas features se desarrollaran de forma modular para facilitar su validación, reutilización y posterior integración en el pipeline de modelado.

Es importante destacar que en esta etapa no se realizan transformaciones dependientes del dataset (como normalización o imputación). Estas se manejarán posteriormente en la etapa de transformación de características, para asegurar consistencia entre entrenamiento e inferencia. El resultado de esta sección será conjunto de variables (X) derivadas más expresivas, listas para ser transformadas y utilizadas por el modelo.

Adicionalmente, las funciones utilizadas en esta sección fueron previamente modularizadas dentro de la carpeta `features/` del proyecto, lo que permite mantener una organización clara y reutilizable. El módulo `build_features` actuará como orquestador de la creación de features, integrando cada módulo en un flujo único y consistente. Este enfoque no solo mejora la calidad de las features, sino que también asegura que el proceso pueda escalar y aplicarse de forma consistente en entornos de entrenamiento e inferencia.

### Features de ubicación (Geo)

La ubicación es uno de los factores más determinantes en el precio de un listing. Sin embargo, variables como latitud y longitud por sí solas no capturan de forma directa el valor contextual de una zona. y no son directamente interpretables por el modelo. 

En esta sección se construyeron features geoespaciales que capturan accesibilidad y atractivo del entorno, como la cercanía a zonas turísticas o culturales y la densidad de actividad comercial en la zona. Con estas variables buscaremos traducir la ubicación en señales más informativas y comparables entre listings.

Entre las variables derivadas se incluyeron:

- `distance_to_attractions`: mide la proximidad mínima de un listing a la atracción cultural o turística más cercana.

- `attractions_density`: cuenta el número de atracciones dentro de un radio definido alrededor del listing, reflejando la concentración cultural o turística.

- `commercial_density`: cuenta el número de puntos de interés comerciales (cafés, restaurantes, bares, etc.) en el mismo radio, como proxy de actividad económica y conveniencia.

Estas variables permitirán traducir coordenadas geográficas en señales más interpretables y útiles para el modelo. Para garantizar reutilización y claridad en el pipeline, la lógica geoespacial fue encapsulada en el módulo `features/geo.py`, donde se implementaron funciones eficientes basadas en estructuras como BallTree para el cálculo de distancias y conteos espaciales. 



In [2]:
# Import the function to add geo features
from src.features.geo import add_distance_to_attractions
from src.features.geo import add_attractions_density
from src.features.geo import add_commercial_density

# Apply the function to df_features to add the new columns
df_features = add_distance_to_attractions(df_features)
df_features = add_attractions_density(df_features)
df_features = add_commercial_density(df_features)

# Display the first rows of latitude, longitude, and the new geo features
geo_features = ['dist_to_nearest_attraction', 'attractions_within_radius', 'commercial_within_radius']
df_features[["latitude", "longitude"] + geo_features]


,latitude,longitude,dist_to_nearest_attraction,attractions_within_radius,commercial_within_radius
0,19.382830,-99.271780,2.450990,0,1
1,19.411620,-99.177940,0.415358,4,252
2,19.439770,-99.156050,0.375601,3,129
3,19.408260,-99.186590,0.985700,1,53
4,19.396750,-99.175810,1.557305,0,116
...,...,...,...,...,...
21490,19.442240,-99.113440,2.178589,0,14
21491,19.308017,-99.168158,0.958782,1,29
21492,19.434460,-99.174010,0.832345,1,128
21493,19.406435,-99.160934,0.958943,1,184


In [3]:
# Validate geo features
validate_features(df_features, geo_features, 'log_price')


========== Validating: dist_to_nearest_attraction ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21495     21495   18081     0           0.0

--- Statistical Summary ---
                          0
mean               1.302349
median             0.716410
mode               0.606966
std                1.588781
min                0.004748
p25                0.394962
p50                0.716410
p75                1.572062
max               16.244343
skew               2.974055
kurt              11.983244
outliers_count  1837.000000
outliers_pct       8.550000
pearson_r         -0.239113

========== Validating: attractions_within_radius ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21495     21495      12     0           0.0

--- Statistical Summary ---
                        0
mean             2.717562
median           2.000000
mode             0.000000
std              2.79

Ahora verificaremos que las variables derivadas hayan sido correctamente construidas y mantengan consistencia con los hallazgos obtenidos durante el análisis exploratorio. Este proceso de creación y verificación se aplicará de manera sistemática a todas las features derivadas.

El objetivo de estas validaciones no será reinterpretar los resultados obtenidos en el EDA, sino asegurar coherencia en la distribución de las variables, ausencia de valores nulos inesperados, comportamiento estadístico consistente y presencia de señal preliminar con la variable objetivo.

1. `dist_to_nearest_attraction`

- La distribución presenta una fuerte asimetría positiva, lo cual es consistente con la naturaleza geográfica del problema (muchas propiedades cercanas a puntos de interés y pocas muy alejadas).
- No se observan valores nulos, lo que confirma que el cálculo de distancias fue aplicado correctamente a todo el dataset.
- La correlación negativa con el precio ("pearson_r ≈ -0.24") es coherente con lo esperado: a mayor distancia, menor valor.
- Se identifican outliers, pero estos representan casos extremos válidos (propiedades alejadas), no errores de cálculo.
- La feature está correctamente construida, con señal consistente y sin anomalías estructurales.

2. `attractions_within_radius`

- Variable discreta con baja cardinalidad, lo cual facilita su interpretabilidad.
- La distribución muestra concentración en valores bajos, indicando que muchas propiedades tienen pocos puntos de interés cercanos.
- La correlación positiva con el precio ("pearson_r ≈ 0.30") confirma la hipótesis inicial: mayor densidad de atracciones → mayor valor.
- No presenta outliers ni valores inconsistentes.
- La feature es estable, interpretable y alineada con los patrones observados en EDA.

3. `commercial_within_radius`

- Presenta una distribución más amplia, reflejando mayor variabilidad en zonas comerciales.
- La correlación positiva ("pearson_r ≈ 0.29") indica que la actividad comercial cercana es un factor relevante en el precio.
- La ausencia de outliers sugiere que la variable captura bien la densidad sin distorsiones extremas.
- Alta cardinalidad, lo que podría requerir transformación posterior dependiendo del modelo.
- Feature con buena señal y variabilidad, candidata fuerte para el modelo, sujeta a posible transformación.

En general, las features de ubicación muestran consistencia con el análisis exploratorio previo y capturan correctamente el contexto espacial de las propiedades, aportando información relevante para el modelado.

### Features de tipo de propiedad

Las variable `property_type` presenta una alta granularidad (por ejemplo, múltiples variantes de apartamentos, casas, habitaciones, etc.), lo que dificulta su uso directo en el modelo y puede introducir ruido.

En esta sección se construyeron las variables derivadas de `property_type` y `room_type` que simplifican y estructuran esta información, manteniendo su significado esencial:

- `property_group`: agrupa los distintos tipos de propiedad en categorías más generales (apartamento, casa, hotel, etc.), reduciendo la cardinalidad y mejorando la interpretabilidad.
- `property_group_room`: combina el grupo de la propiedad con el tipo de habitación, capturando interacciones relevantes entre ambos factores.

Esta segunda variable resulta especialmente útil, ya que permite diferenciar escenarios que, aunque comparten el mismo tipo de propiedad, tienen comportamientos distintos en función del tipo de ocupación. El objetivo de estas creaciones fue transformar variables categóricas complejas en representaciones más compactas y expresivas, facilitando su posterior codificación y uso en el modelo.

La lógica de estas transformaciones fue modularizada en el modulo `features/property.py`, permitiendo su reutilización dentro del pipeline de `build_features`.




In [4]:
# Import the function to add property features
from src.features.property import add_property_group
from src.features.property import add_property_group_room

# Apply the function to df_features to add the new columns
df_features = add_property_group(df_features)
df_features = add_property_group_room(df_features)

# Display the first rows of property features
property_features = ['room_type','property_group', 'property_group_room']
df_features[property_features]

,room_type,property_group,property_group_room
0,entire_home/apt,house,house_entire_home/apt
1,entire_home/apt,house,house_entire_home/apt
2,entire_home/apt,apartment,apartment_entire_home/apt
3,entire_home/apt,house,house_entire_home/apt
4,private_room,apartment,apartment_private_room
...,...,...,...
21490,private_room,apartment,apartment_private_room
21491,private_room,apartment,apartment_private_room
21492,entire_home/apt,apartment,apartment_entire_home/apt
21493,entire_home/apt,apartment,apartment_entire_home/apt


In [5]:
# Validate property features
validate_features(df_features, property_features, 'log_price')


========== Validating: room_type ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21495     21495       4     0           0.0

--- Frequency distribution ---
                 count    pct
room_type                    
entire_home/apt  14783  68.77
private_room      6433  29.93
shared_room        232   1.08
hotel_room          47   0.22

--- Median target by category ---
         room_type  median_target
0       hotel_room       7.600902
1  entire_home/apt       7.133296
2     private_room       6.363028
3      shared_room       5.579730

========== Validating: property_group ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21495     21495       6     0           0.0

--- Frequency distribution ---
                count    pct
property_group              
apartment       16604  77.25
house            2881  13.40
hotel             992   4.62
guesthouse        804   3.74
unique/n

1. `room_type`

- Variable categórica con baja cardinalidad (4 categorías) y sin valores nulos.
- La distribución está claramente desbalanceada, con predominancia de propiedades completas (~69%), seguida de habitaciones privadas (~30%).
- La variable captura bien diferencias de precio entre modalidades de alojamiento, con un gradiente claro de hotel > entire home > private room > shared room.
- Feature estable, con buena interpretabilidad y señal clara, alineada con la lógica del negocio.

2. `property_group`

- La agrupación reduce efectivamente la cardinalidad del tipo de propiedad original a 6 categorías.
- La distribución está dominada por apartamentos (~77%), lo cual refleja la composición del dataset.
- Se mantiene una diferenciación en términos de precio entre categorías, aunque con menor separación que en `room_type`.
- Feature correctamente agregada, útil para reducir complejidad sin perder completamente la señal original.

3. `property_group_room`

- Variable resultante de la interacción entre `property_group` y `room_type`, con mayor granularidad (19 categorías).
- Permite capturar diferencias relevantes dentro de una misma categoría de propiedad (por ejemplo, propiedad completa vs. habitación privada dentro de apartamentos).
- Se observa una mayor dispersión en los valores del target, lo que sugiere que esta variable captura interacciones más específicas.
- Algunas categorías presentan baja frecuencia, lo cual podría requerir tratamiento dependiendo del modelo.
- Feature más expresiva que las anteriores, con potencial para capturar relaciones más complejas, sujeta a evaluación posterior en la etapa de selección de variables.

En conjunto, las variables de tipo de propiedad muestran consistencia estructural, ausencia de anomalías y una relación coherente con la variable objetivo, quedando listas para su posterior transformación y evaluación dentro del modelo.

### Features de restricciones de reserva

Las variables relacionadas con las restricciones de reserva pueden reflejar el tipo de estancia que un anfitrión busca atraer, lo cual puede influir indirectamente en el precio de la propiedad. Durante el análisis exploratorio se evaluaron tanto `minimum_nights` como `maximum_nights`. Sin embargo, se observó que `maximum_nights` no aportaba información relevante para explicar la variación del precio, por lo que se decidió no incluirla en el proceso de *feature engineering*.

En cambio, `minimum_nights` mostró patrones más claros, por lo que se utilizó para construir una nueva variable categórica:

- `minimum_nights_segment`: agrupa las propiedades en distintos tipos de estancia (corta, media y larga duración), utilizando rangos definidos que representan comportamientos típicos de reserva.

Esta transformación permitió capturar de forma más interpretable la intención del anfitrión en términos de duración de estancia, simplificando una variable numérica en categorías con significado práctico y facilitando su uso posterior en el modelo.

La lógica de esta transformación fue encapsulada en el módulo `features/booking_restrictions.py`, permitiendo su reutilización dentro del pipeline de `build_features`.

In [6]:
# Import the function to add booking restrictions features
from src.features.booking_restrictions import add_minimum_nights_segment

# Apply the function to df_features to add the new column
df_features = add_minimum_nights_segment(df_features)

# Display the first rows of booking restrictions features
booking_restrictions_features = ['minimum_nights', 'minimum_nights_segment']
df_features[booking_restrictions_features]

,minimum_nights,minimum_nights_segment
0,1,short_stay
1,1,short_stay
2,15,medium_stay
3,2,short_stay
4,4,medium_stay
...,...,...
21490,1,short_stay
21491,1,short_stay
21492,1,short_stay
21493,1,short_stay


In [7]:
# Validate booking restrictions features
validate_features(df_features, booking_restrictions_features, 'log_price')


========== Validating: minimum_nights ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21495     21495      52     0           0.0

--- Statistical Summary ---
                          0
mean               3.171110
median             1.000000
mode               1.000000
std               13.396934
min                1.000000
p25                1.000000
p50                1.000000
p75                2.000000
max              365.000000
skew              20.920417
kurt             522.484631
outliers_count  2180.000000
outliers_pct      10.140000
pearson_r          0.029767

========== Validating: minimum_nights_segment ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0  category  21495     21495       3     0           0.0

--- Frequency distribution ---
                        count    pct
minimum_nights_segment              
short_stay              19315  89.86
medium_stay              1

1. `minimum_nights`

- Variable numérica con alta asimetría positiva, lo cual es consistente con su naturaleza: la mayoría de las propiedades permiten estancias cortas, mientras que existen pocos casos con valores muy altos.
- Se observan valores extremos (hasta 365 noches), lo que genera una alta curtosis y presencia de outliers (~10%). Estos valores, aunque extremos, representan casos válidos dentro del contexto del problema.
- La correlación con el precio es prácticamente nula ("pearson_r ≈ 0.03"), lo que indica que la variable en su forma original no captura una relación clara con la variable objetivo.
- Feature variable con distribución altamente sesgada y baja señal directa, lo que justifica su transformación.

2. `minimum_nights_segment`

- Variable categórica con baja cardinalidad (3 categorías) y sin valores nulos.
- La distribución está fuertemente concentrada en estancias cortas (~90%), reflejando el comportamiento predominante del mercado.
- Se observa una diferenciación en términos de precio entre segmentos, lo que sugiere que la agrupación captura patrones que no eran evidentes en la variable original.
- La transformación reduce la influencia de valores extremos y facilita la interpretación.

### Features de amenidades y servicios

Las amenidades y servicios disponibles en una propiedad representan uno de los componentes mas relevantes en la percepción de valor por parte de los usuarios, ya que reflejan el nivel de confort, funcionalidad y experiencia ofrecida.

Durante el análisis exploratorio se identificó que la información contenida en la lista de amenidades tiene un alto potencial predictivo, aunque se encuentra en un formato no estructurado. Por ello, en esta sección se construyeron diversas variables derivadas que permitieron transformar esta información en representaciones más útiles para el modelo.

Este enfoque permite capturar tanto información global como específica sobre las amenidades, combinando simplicidad e interpretabilidad con mayor capacidad de representación.

Las principales transformaciones incluyen:

- `amenities_count`:  medida general del nivel de equipamiento.
- `has_amenity`: columnas bonarias que indican la presencia de amenidades específicas relevantes.
- `amenities_score`: resumen el impacto combinado de múltiples amenidades en una sola métrica.

La lógica de estas transformaciones fue encapsulada en el módulo dedicado `amenities.py`, asegurando su reutilización dentro del pipeline de `build_features`.

#### Creación de `amenity_count`

Como primer paso en la creación de variables relacionadas con amenidades, se construyó una métrica simple pero informativa:

- `amenities_count`: representa el número total de amenidades disponibles en cada propiedad, calculado a partir de la lista original.

Esta variable permite capturar el nivel general de equipamiento de forma directa, transformando información no estructurada en una señal cuantitativa interpretable. Sin embargo, dado que la relación entre el número de amenidades y el precio presenta una relación lineal moderada, se contempló y analizó una versión discretizada de esta variable en el análisis exploratorio, la cual será generada en la etapa de transformación de características.

Con la creación de `amenity_count` establecemos una base cuantitativa sobre la cual se construirán representaciones más específicas del impacto de las amenidades.

In [8]:
# Import the function to add amenitiy_count feature
from src.features.amenities import add_amenity_count

# Apply the function to df_features to add the new column
df_features = add_amenity_count(df_features)

# Display the first rows of amenity_count
df_features[['amenities', 'amenity_count']]

,amenities,amenity_count
0,"[""Garden view"", ""Resort access"", ""Washer"", ""Co...",12
1,"[""Piano"", ""Patio or balcony"", ""Wifi"", ""Refrige...",26
2,"[""Self check-in"", ""Dining table"", ""Elevator"", ...",28
3,"[""Pack \u2019n play/Travel crib"", ""Self check-...",47
4,"[""Elevator"", ""Wifi"", ""Hot water"", ""Microwave"",...",24
...,...,...
21490,"[""Private entrance"", ""Laundromat nearby"", ""Din...",28
21491,"[""Washer"", ""Iron"", ""Kitchen"", ""Clothing storag...",9
21492,"[""Paid parking off premises"", ""Private entranc...",17
21493,"[""Exterior security cameras on property"", ""Kit...",11


In [9]:
# Validate amenity_count feature
validate_features(df_features, ['amenity_count'], 'log_price')


========== Validating: amenity_count ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21495     21495      92     0           0.0

--- Statistical Summary ---
                         0
mean             33.147383
median           34.000000
mode             35.000000
std              15.537351
min               0.000000
p25              22.000000
p50              34.000000
p75              44.000000
max             102.000000
skew              0.070188
kurt             -0.428441
outliers_count   68.000000
outliers_pct      0.320000
pearson_r         0.335379


1. Perfil general de `amenity_count`

- Variable numérica sin valores nulos, lo que confirma que el proceso de parsing y conteo fue aplicado correctamente a todo el dataset.
- En algunos listings se obtuvieron listas vacías [], las cuales se traducen en un conteo igual a 0, indicando propiedades sin amenidades declaradas.
- Presenta una distribución relativamente equilibrada, con baja asimetría *(skew ≈ 0.07)* y curtosis cercana a una distribución normal.
- La dispersión es moderada, reflejando una variabilidad razonable en el nivel de equipamiento entre propiedades.
- Se identifican pocos valores extremos (~0.3%), lo que sugiere una distribución estable y sin anomalías significativas.

2. Relación con la variable objetivo

- La correlación positiva con el precio (*pearson_r ≈ 0.34*) indica una relación consistente entre mayor nivel de equipamiento y mayor valor de la propiedad.
- Esta señal resulta especialmente relevante considerando que se trata de una métrica agregada simple.

La variable `amenities_count` presenta una estructura estadística sólida, buena cobertura y una señal preliminar clara respecto a la variable objetivo: la cantidad total de amenidades si captura valor real. Su comportamiento sugiere que constituye una base robusta para representar el nivel general de equipamiento, además de servir como insumo para transformaciones posteriores como su discretización en bins.

#### Creación de columnas binarias `has_amenity`

Si bien el conteo total de amenidades permite capturar el nivel general de equipamiento, no todas las amenidades aportan el mismo valor. Algunas tienen un impacto más significativo en la percepción del usuario y, potencialmente, en el precio.

Por esta razón, en esta sección se construyeron variables binarias del tipo:

- "has_amenity": indican la presencia o ausencia de amenidades específicas dentro de cada propiedad.

Estas variables permiten modelar el efecto individual de servicios concretos (por ejemplo, aire acondicionado, estacionamiento, televisión, etc.), capturando información que se pierde al utilizar únicamente métricas agregadas.

Para su construcción, se realizó un proceso de limpieza y normalización de la lista de amenidades, seguido de la identificación de un conjunto de servicios relevantes. A partir de este conjunto, se generaron columnas binarias que reflejan la disponibilidad de cada amenidad.

In [10]:
# Import the function to add has_amenity features
from src.features.amenities import add_has_amenity_features

# Apply the function to df_features to add the new columns
df_features, has_amenity_cols = add_has_amenity_features(df_features)

# Display the first rows of has_amenity columns
df_features[has_amenity_cols]

,has_wifi,has_kitchen,has_hot_water,has_essentials,has_bed_linens,has_microwave,has_refrigerator,has_air_conditioning,has_heating,has_cooking_basics,...,has_backyard,has_sauna,has_city_skyline_view,has_outdoor_furniture,has_outdoor_dining_area,has_smoke_alarm,has_carbon_monoxide_alarm,has_fire_extinguisher,has_exterior_security_cameras_on_property,has_first_aid_kit
0,True,True,True,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,True,True,False,True,False,True,True,False,False,True,...,True,False,False,False,False,True,True,False,True,False
2,True,True,True,True,True,True,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,True,True,True,True,True,True,True,False,True,True,...,True,False,False,True,True,False,False,False,False,True
4,True,True,True,True,False,True,True,False,False,True,...,False,False,False,False,False,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21490,True,True,True,False,True,True,True,False,False,True,...,False,False,False,False,False,True,True,False,True,False
21491,True,True,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
21492,True,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,True,True,False,False,False
21493,True,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,True,True,True,True


In [11]:

# Validate has_amenity features

# 1. Initialize a list to store results
results = []

# 2. Compute amenity-level statistics:
# For each binary amenity feature, calculate prevalence, median price differences,
# relative impact on price, and correlation with log_price.
for col in has_amenity_cols:
    # Proportion of listings that have the amenity
    proportion = df_features[col].mean()
    
    # Median log_price for listings with and without the amenity
    median_true = df_features.loc[df_features[col], "log_price"].median()
    median_false = df_features.loc[~df_features[col], "log_price"].median()
    
    # Difference in medians (impact of having the amenity, en log)
    median_diff = median_true - median_false
    
    # Relative differences in price (%)
    real_median_diff = np.exp(median_diff) - 1

    # Correlation between amenity (binary) and log_price
    corr = df_features[col].corr(df_features["log_price"])
    
    # Append results as a dictionary
    results.append({
        "feature": col,
        "proportion": proportion,
        "median_true": median_true,
        "median_false": median_false,
        "median_diff": median_diff,
        "real_median_diff": f"{real_median_diff*100:.3f}%",
        "correlation_r": corr
    })

# 3. Convert results into a DataFrame
impact_has_amenity = pd.DataFrame(results).set_index("feature")

# 4. Display the final table
impact_has_amenity.sort_values(by="proportion", ascending=False).round(3)

,proportion,median_true,median_false,median_diff,real_median_diff,correlation_r
feature,,,,,,
has_wifi,0.990,6.957,6.867,0.090,9.375%,-0.001
has_kitchen,0.892,6.985,6.667,0.318,37.405%,0.085
has_tv,0.861,7.026,6.322,0.704,102.156%,0.285
has_hot_water,0.844,6.987,6.712,0.276,31.752%,0.097
has_cooking_basics,0.751,7.014,6.709,0.305,35.610%,0.137
has_dishes_and_silverware,0.743,7.006,6.745,0.261,29.765%,0.125
has_iron,0.742,7.032,6.667,0.365,44.020%,0.191
has_essentials,0.717,7.015,6.763,0.252,28.671%,0.140
has_dedicated_workspace,0.702,6.992,6.883,0.109,11.475%,0.072


Dado el número de variables generadas, se utiliza un enfoque agregado para evaluar su calidad, considerando métricas como proporción, diferencias en la variable objetivo y correlación.

1. Perfil general

- Las variables presentan proporciones diversas, desde amenidades casi universales (por ejemplo, `has_wifi`) hasta otras poco frecuentes (por ejemplo, `has_piano`, `has_sauna`).
- Esta variabilidad en frecuencia es esperada y refleja la heterogeneidad en el nivel de equipamiento entre propiedades.
- No se observan inconsistencias estructurales en la construcción de las variables.

2. Relación con la variable objetivo

- En general, múltiples amenidades muestran diferencias positivas en la mediana del precio cuando están presentes, lo que sugiere una contribución potencial al valor de la propiedad.
- Algunas variables presentan correlaciones moderadas (por ejemplo, `has_tv`, `has_hair_dryer`, `has_elevator`, `has_free_parking`), indicando señal consistente.
- También se identifican amenidades con bajo o nulo impacto aparente, así como algunos casos con diferencias negativas, lo cual es esperable dada la diversidad y frecuencia de estas variables.

3. Consideraciones

- Amenidades con proporciones muy altas (cercanas a 1) o muy bajas (cercanas a 0) tienden a aportar menor información efectiva al modelo.
- Algunas variables con alto impacto aparente pueden estar influenciadas por baja frecuencia, por lo que su efecto deberá evaluarse con mayor cuidado.
- - A partir de estos criterios, se considera selccionar un subconjunto de variables que combine relevancia estadística, interpretabilidad y frecuencia suficiente, con el objetivo de maximizar la señal predictiva y reducir el riesgo de ruido, redundancia o sobreajuste.

Las variables binarias "has_amenity" fueron correctamente construidas y presentan un comportamiento coherente con lo observado durante el análisis exploratorio. Estas variables permiten capturar señales específicas que complementan las métricas agregadas, y serán evaluadas de forma conjunta en la etapa de selección de variables y construcción del modelo.

#### Creación de `amenity_score`

Las variables binarias (`has_amenity`) permiten capturar el efecto individual de cada servicio, pero su uso directo puede introducir alta dimensionalidad y fragmentación de la señal.

Para abordar esto, en esta sección se construyó una variable agregada:

- `amenity_score`: score numérico que resume el impacto conjunto de múltiples amenidades en una sola métrica.

El proceso de construcción del score sigue tres pasos principales:

- Selección de amenidades relevantes, basada en criterios de frecuencia y relación con la variable objetivo.
- Asignación de pesos, calculados como la combinación de la diferencia de medianas y la correlación (*median_diff * correlation*). Se multiplican ambas métricas para obtener un peso que refleja tanto el impacto como la consistencia estadística de la relación.
- Cálculo del score, mediante la suma de los pesos de las amenidades presentes en cada propiedad.

$$
\text{amenity\_score}_{listing} = \sum_{i=1}^{n} \big( \text{has\_amenity}_{i} \cdot w_{\text{amenidad i}} \big)
$$


De esta forma, `amenitiy_score` permite consolidar múltiples señales en una representación más compacta y expresiva. Reduciendo dimensionalidad sin perder información relevante, capturando el valor agregado de las amenidades de forma más eficiente.

In [12]:
# Import the function to add amenity_score feature
from src.features.amenities import add_amenity_score

# Apply the function to df_features to add the new columns
df_features = add_amenity_score(df_features)

# Display the first rows of has_amenities columns
df_features['amenity_score']

0        0.268380
1        1.044497
2        0.976673
3        1.261502
4        1.033924
           ...   
21490    0.649838
21491    0.409493
21492    0.693755
21493    0.286073
21494    0.286073
Name: amenity_score, Length: 21495, dtype: float64

In [13]:
# Validate amenity_score feature
validate_features(df_features, ['amenity_score'], 'log_price')


========== Validating: amenity_score ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21495     21495   11704     0           0.0

--- Statistical Summary ---
                       0
mean            0.937367
median          1.001418
mode            0.227788
std             0.388871
min             0.000000
p25             0.662696
p50             1.001418
p75             1.234661
max             1.874167
skew           -0.400404
kurt           -0.628622
outliers_count  0.000000
outliers_pct    0.000000
pearson_r       0.429282


1. Perfil estadístico

- Variable numérica sin valores nulos, lo que confirma la correcta aplicación del proceso de agregación.
- Presenta una distribución relativamente equilibrada, con ligera asimetría negativa (*skew ≈ -0.40*), indicando una mayor concentración de propiedades con scores altos.
- La dispersión es moderada, permitiendo diferenciar adecuadamente entre distintos niveles de equipamiento.
- No se observan outliers, lo que sugiere que el proceso de construcción del score es estable y robusto.

2. Relación con la variable objetivo

- La correlación con el precio (*pearson_r ≈ 0.43*) es considerablemente más alta que la observada en variables individuales como `amenities_count` o las variables binarias.
- Esto indica que el score logra capturar de manera más efectiva el impacto conjunto de las amenidades en el valor de la propiedad.

La variable `amenities_score` presenta una estructura estadística sólida y una señal clara respecto a la variable objetivo. Su comportamiento sugiere que esta variable logra consolidar información dispersa en una representación más compacta y expresiva, convirtiéndose en una de las features más prometedoras dentro del conjunto de amenidades.

### Features del anfitrión

Las características del anfitrión pueden influir en la percepción de confianza y calidad por parte de los usuarios, lo que potencialmente puede impactar en el precio de la propiedad.

En esta sección se construyó una variable derivada a partir de la información de verificaciones del anfitrión:

- `host_verifications_grouped`: agrupa los distintos tipos de verificación en categorías más interpretables (por ejemplo, sin verificación, básica, extendida).

Dado que la variable original (`host_verifications`) se encuentra en un formato no estructurado, se realizó un proceso de transformación para convertirla en una representación categórica más compacta. Esta agrupación permite capturar distintos niveles de verificación y confianza del anfitrión de forma más clara, lo que también facilita su uso posterior en el modelo.

La lógica de esta transformación fue encapsulada en el módulo dedicado `features/host.py`, permitiendo su reutilización dentro del pipeline de `build_features`.

In [14]:
# Import the function to add amenity_score feature
from src.features.host import add_host_verifications_grouped

# Apply the function to df_features to add the new columns
df_features = add_host_verifications_grouped(df_features)

# Display the first rows of has_amenities columns
df_features[['host_verifications', 'host_verifications_grouped']]

,host_verifications,host_verifications_grouped
0,"['email', 'phone', 'work_email']",extended
1,"['email', 'phone', 'work_email']",extended
2,"['email', 'phone']",basic
3,"['email', 'phone']",basic
4,"['email', 'phone']",basic
...,...,...
21490,"['email', 'phone']",basic
21491,"['email', 'phone']",basic
21492,"['email', 'phone']",basic
21493,['phone'],low


In [15]:
# Validate host_verifications_grouped
validate_features(df_features, ['host_verifications_grouped'], 'log_price')


========== Validating: host_verifications_grouped ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21495     21495       4     0           0.0

--- Frequency distribution ---
                            count    pct
host_verifications_grouped              
basic                       16721  77.79
extended                     2690  12.51
low                          2072   9.64
none                           12   0.06

--- Median target by category ---
  host_verifications_grouped  median_target
0                   extended       7.102499
1                      basic       6.937314
2                        low       6.891116
3                       none       6.735312


1. Perfil general

- Variable categórica sin valores nulos y con baja cardinalidad (4 categorías), lo que facilita su interpretación y uso en modelos.
- La distribución está fuertemente concentrada en la categoría "basic" (~78%), seguida por "extended" y "low", mientras que "none" representa una proporción marginal.
- Esta distribución es consistente con el comportamiento esperado en plataformas donde la mayoría de anfitriones cuentan con verificaciones básicas.

2. Relación con la variable objetivo

- Se observa una tendencia creciente en el precio conforme aumenta el nivel de verificación:
  - *extended* presenta la mediana más alta.
  - *basic* y "low" muestran valores intermedios.
  - *none* presenta la mediana más baja.
- Esto sugiere que un mayor nivel de verificación podría estar asociado con propiedades de mayor valor.

3. Consideraciones

- La categoría "none" tiene una frecuencia extremadamente baja, por lo que su comportamiento debe interpretarse con cautela.
- Las diferencias entre categorías son moderadas, lo que indica que la señal existe, pero no es dominante frente a otras variables del modelo.

La variable `host_verifications_grouped` fue correctamente construida y presenta un comportamiento coherente con lo esperado. Aunque su impacto individual es limitado, aporta una señal contextual relacionada con la confianza y perfil del anfitrión, lo que podría complementar otras variables dentro del modelo.

### Features de puntuaciones de reseñas

Las puntuaciones de reseñas reflejan la experiencia de los usuarios en una propiedad y constituyen una señal directa de calidad percibida lo que podría incluir en el precio.

En esta sección se construyeron dos variables derivadas a partir de las distintas métricas de evaluación disponibles en el conjunto de datos:

- `review_scores_mean`: promedio de las diferentes dimensiones de evaluación (limpieza, comunicación, ubicación, etc.), con el objetivo de obtener una medida global de calidad.
- `has_review`: variable binaria que indica si la propiedad cuenta o no con reseñas.

La variable `review_scores_mean` permite consolidar múltiples dimensiones en una representación única y más manejable, mientras que `has_review` captura la diferencia entre propiedades con historial de evaluaciones y aquellas sin información disponible.

Estas transformaciones permitirán incorporar tanto la calidad percibida como la disponibilidad de información, dos factores relevantes en la valoración de una propiedad.

La lógica de construcción fue modularizada en `features/reviews.py` para su integración dentro del pipeline de `build_features`.



In [16]:
# Import the function to add review features
from src.features.reviews import add_review_scores_mean
from src.features.reviews import add_has_review

# Apply the function to df_features to add the new columns
df_features = add_review_scores_mean(df_features)
df_features = add_has_review(df_features)

# Display the first rows of review features
review_features = ['review_scores_mean', 'has_review']
df_features[review_features]

,review_scores_mean,has_review
0,NaN,False
1,4.707143,True
2,4.881429,True
3,4.868571,True
4,4.862857,True
...,...,...
21490,NaN,False
21491,NaN,False
21492,NaN,False
21493,NaN,False


In [17]:
validate_features(df_features, review_features, 'log_price')


========== Validating: review_scores_mean ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  19010     19010    1167  2485         11.56

--- Statistical Summary ---
                          0
mean               4.798076
median             4.857143
mode               5.000000
std                0.275122
min                1.000000
p25                4.761429
p50                4.857143
p75                4.925714
max                5.000000
skew              -6.290170
kurt              60.285499
outliers_count  1178.000000
outliers_pct       6.200000
pearson_r          0.131218

========== Validating: has_review ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0      bool  21495     21495       2     0           0.0

--- Frequency distribution ---
            count    pct
has_review              
True        19010  88.44
False        2485  11.56

--- Median target by category ---
   has_rev

1. `review_scores_mean`

- La variable presenta valores faltantes (~11.6%), correspondientes a propiedades sin reseñas.
- La distribución está fuertemente sesgada hacia valores altos (~*skew ≈ -6.29*~), con una alta concentración cercana al máximo (5.0).
- La baja dispersión (*std ≈ 0.27*) indica que la mayoría de las propiedades tienen evaluaciones similares.
- Se observan outliers en la cola baja, correspondientes a propiedades con calificaciones significativamente menores.

Respecto a la relación con la variable objetivo:

- La correlación con el precio es baja (*pearson_r ≈ 0.13*), lo que sugiere una relación limitada.
- Esto puede explicarse por la falta de variabilidad en la variable, ya que la mayoría de las propiedades tienen puntuaciones altas.


2. `has_review`

- Variable binaria sin valores nulos.
- La mayoría de las propiedades cuentan con reseñas (~88%), mientras que una proporción menor (~12%) no tiene información disponible.
- Se observa que las propiedades sin reseñas presentan una mediana de precio ligeramente superior a aquellas con reseñas.
- Este comportamiento puede estar influenciado por factores externos, como propiedades nuevas.

3. Consideraciones

- La variable `review_scores_mean` presenta saturación en valores altos, lo que limita su capacidad discriminativa.
- La señal observada en `has_review` no necesariamente implica causalidad directa, sino que puede reflejar efectos indirectos del mercado.

Las variables de reseñas fueron correctamente construidas y aportan información complementaria sobre la calidad percibida y la disponibilidad de evaluaciones. Sin embargo, su capacidad predictiva individual parece limitada, por lo que su utilidad deberá evaluarse en conjunto con el resto de variables durante la etapa de modelado.

## Transformación de Características (Transform Features)

Una vez construidas las variables derivadas, el siguiente paso consiste en transformar las características para hacerlas más adecuadas para el modelado. Esta etapa busca ajustar la representación de los datos con el fin de mejorar la estabilidad, comparabilidad y capacidad predictiva de las variables.

Las transformaciones pueden incluir ajustes en la escala, tratamiento de distribuciones, manejo de valores faltantes, manejo de valores extremos o cambios en la representación de las variables, dependiendo de su naturaleza.

Para organizar este proceso de manera clara y reproducible, la sección se dividirá en dos partes principales:

- **Transformación de variables numéricas**
- **Transformación de variables categóricas**

Cada grupo será abordado de forma independiente, aplicando técnicas específicas acordes a sus características y comportamiento en los datos.

### Evaluación de variables numéricas

En esta sección se abordará el diagnóstico y evaluación de las variables numéricas, con el propósito de identificar sus características estadísticas y definir el tratamiento más adecuado para cada una.

Para ello, se seguirá un enfoque estructurado que combina dos etapas principales: **diagnóstico** y **comparación de transformaciones**. A partir del diagnóstico estadístico de cada variable, se analizan aspectos como la presencia de valores nulos, la forma de la distribución, la existencia de valores extremos y su relación con la variable objetivo.

Con base en este análisis, se generan sugerencias de tratamiento que permiten establecer, para cada variable:

- El método de imputación de valores nulos  
- La necesidad (o no) de aplicar transformaciones como logaritmos o winsorización  
- La conveniencia de realizar agrupaciones en bins para capturar posibles relaciones no lineales  
- El tipo de escalado más adecuado para su uso en modelos  

Adicionalmente, se realizará una comparación entre distintas transformaciones para aquellas variables que lo requieren, con el fin de evaluar su impacto y seleccionar la representación más adecuada.

A partir de estos resultados se determinarán las estrategias de imputación, transformación y escalado, con el fin de optimizar la representación de las variables y mejorar su utilidad en el modelo predictivo. Este enfoque permitirá tomar decisiones informadas y consistentes, evitando aplicar transformaciones de forma arbitraria y asegurando que cada variable sea tratada según su comportamiento específico en los datos.

#### Diagnóstico de variables

In [172]:
# Import diagnostic and comparison functions
from src.features.diagnose_features import build_numeric_diagnostics
from src.features.compare_features import compare_transformations

# Define numeric features to analyze (including target)
numeric_features = [
    # numeric features
    'dist_to_nearest_attraction', 'attractions_within_radius', 'commercial_within_radius',
    'accommodates', 'bathrooms', 'bedrooms', 'beds','amenity_count', 'amenity_score',
    'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 
    'host_total_listings_count', 'review_scores_mean', 'minimum_nights',
    # target
    'log_price'
]

# Create a copy of the DataFrame with selected numeric features
df_numeric_features = df_features[numeric_features].copy()

# Display diagnostics for numeric features (skewness, outliers, correlation, etc.)
print("\n ======================== DIAGNOSTIC OF NUMERICAL FEATURES ========================\n")
build_numeric_diagnostics(df_numeric_features)



 ======================== DIAGNOSTIC OF NUMERICAL FEATURES ========================



,feature,nulls_%,skew,kurtosis,std,min,max,outliers_%,corr_with_target,signal_suggestion,null_treatment_suggestion,transform_suggestion,binning_suggestion,scaling_suggestion
0,dist_to_nearest_attraction,0.000000,2.974055,11.983244,1.588781,0.004748,16.244343,0.085462,-0.239113,useful_signal,no_impute,log,no_binning,robust
1,attractions_within_radius,0.000000,0.677615,-0.615463,2.799745,0.000000,11.000000,0.000000,0.304163,useful_signal,no_impute,no_transform,no_binning,minmax
2,commercial_within_radius,0.000000,0.662185,-0.568576,142.322684,0.000000,552.000000,0.000000,0.294451,useful_signal,no_impute,no_transform,no_binning,minmax
3,accommodates,0.000000,2.502851,9.180266,2.293873,1.000000,16.000000,0.048616,0.591000,useful_signal,no_impute,log,no_binning,minmax
4,bathrooms,0.000558,4.652531,46.499615,0.895753,0.000000,21.000000,0.025462,0.386998,useful_signal,median,log,binning_candidate (low_variance/discrete),minmax
5,bedrooms,0.004513,6.196059,119.668043,1.047044,0.000000,40.000000,0.031078,0.501406,useful_signal,median,log,no_binning,minmax
6,beds,0.000744,4.895545,52.955462,1.584790,0.000000,40.000000,0.095861,0.443860,useful_signal,median,log,no_binning,robust
7,amenity_count,0.000000,0.070188,-0.428441,15.537351,0.000000,102.000000,0.003164,0.335379,useful_signal,no_impute,no_transform,no_binning,standard
8,amenity_score,0.000000,-0.400404,-0.628622,0.388871,0.000000,1.874167,0.000000,0.429282,useful_signal,no_impute,no_transform,binning_candidate (low_variance/discrete),standard
9,calculated_host_listings_count_entire_homes,0.000000,4.123914,18.294074,33.677862,0.000000,221.000000,0.143522,0.198570,useful_signal,no_impute,log,binning_candidate (extreme_values),robust


El diagnóstico realizado permite identificar patrones clave en el comportamiento de las variables numéricas, así como definir estrategias de transformación coherentes para su uso en el modelado.

**1. Presencia de señal en la mayoría de variables**

La gran mayoría de las variables numéricas presentan una correlación relevante con variable objetivo (`log_price`)  clasificándose como **useful_signal**. Este hallazgo confirma que el conjunto de variables no está compuesto por atributos irrelevantes, sino por características con potencial explicativo y predictivo. La diversidad de fuentes de señal (capacidad, ubicación, amenidades, host) también sugiere que el modelo podrá capturar distintos aspectos del fenómeno de precios.

**Excepciones identificadas:**
- `minimum_nights` → presenta una correlación prácticamente nula con el target (≈ 0.03), lo que indica que su aporte directo es limitado.  
- `host_total_listings_count` y `review_scores_mean` → muestran correlaciones débiles (≈ 0.13), clasificándose como **weak_signal**.  

Estas variables, aunque no deben descartarse de inmediato, requieren un tratamiento más cuidadoso. Su valor puede emerger tras transformaciones (logarítmicas, binning) o al ser combinadas con otras señales. Sin embargo, también se podría reconsiderar su inclusión en el modelo si tras pruebas no aportan mejora significativa, evitando así ruido o sobreajuste.

**2. Alta asimetría y presencia de outliers**

El diagnóstico revela un patrón consistente en múltiples variables caracterizado por:

- **Alta skewness (asimetría positiva):** distribuciones sesgadas hacia la derecha, donde la mayoría de observaciones se concentran en valores bajos y existen colas largas hacia valores altos.
- **Alta kurtosis:** presencia de colas pesadas y picos pronunciados, lo que indica una mayor probabilidad de valores extremos.
- **Outliers significativos:** proporciones relevantes de observaciones que se alejan del rango central, afectando la estabilidad estadística y el comportamiento de los modelos.

Este comportamiento es típico de variables con **distribuciones de cola larga**, donde predominan valores pequeños pero aparecen casos aislados con magnitudes muy superiores. Dicho patrón puede distorsionar coeficientes en modelos lineales, afectar métricas de distancia en algoritmos basados en similitud y ralentizar la convergencia en métodos de optimización.

Las variables más afectadas son:

- **Variables de capacidad:**  
  - `accommodates`, `beds`, `bedrooms`, `bathrooms`  
  Presentan alta asimetría y curtosis extrema, reflejando que la mayoría de alojamientos tienen valores bajos, pero existen casos con capacidades muy elevadas que generan colas largas.

- **Variables relacionadas con hosts:**  
  - `calculated_host_listings_count_*`, `host_total_listings_count`  
  Exhiben distribuciones altamente sesgadas, con la mayoría de hosts gestionando pocos anuncios y unos pocos con cientos de propiedades, lo que introduce valores extremos.

- **Restricciones de estancia:**  
  - `minimum_nights`  
  Es el caso más extremo, con skew ≈ 20 y curtosis > 500. La mayoría de listados tienen estancias mínimas cortas, pero algunos presentan valores desproporcionadamente altos (ej. cientos de noches), generando una distribución altamente no lineal.

**Implicaciones y tratamiento sugerido:**
- Aplicar **transformaciones logarítmicas** para reducir la asimetría y estabilizar la varianza.  
- Considerar **escalado robusto** (basado en mediana y rango intercuartílico) para mitigar el impacto de outliers.  
- Evaluar **binning** en variables discretas o con valores extremos recurrentes, como `bathrooms` o `minimum_nights`.  
- Validar el efecto de estas transformaciones en la correlación con el target, asegurando que la señal útil se preserve o incluso se potencie.

La alta asimetría y presencia de outliers es un rasgo estructural de varias variables. Su tratamiento adecuado será esencial para garantizar estabilidad en el modelado y evitar que valores extremos dominen el aprendizaje.

**3. Transformaciones logarítmicas como solución clave**

Dado el comportamiento anterior identificado en múltiples variables hace que la **transformación logarítmica (`log`)** se convierta en una herramienta fundamental dentro del pipeline de preprocesamiento. La transformación logaritmica resulta adecuada para:

- Reducir la asimetría, suavizar distribuciones sesgadas y acercárlas a formas más simétricas. 
- Comprimir valores extremos, los outliers pierden influencia relativa disminuyendo su impacto en el modelo.
- Linealizar relaciones, facilitando que las variables se relacionen de manera más proporcional con el target (`log_price`)
- Estabilizar la varianza, mejora la convergencia de algoritmos y reduce el riesgo de sobreajuste.

**Consideraciones prácticas:**
- Para evitar problemas con valores cero, se podría utilizar `log1p(x)` en lugar de `log(x)`, asegurando estabilidad en todas las observaciones.  
- La transformación debe validarse comparando correlaciones y ajustes antes y después, confirmando que la señal con el target se mantiene o mejora.  
- En algunos casos, la combinación de log con **escalado robusto** ofrece mayor estabilidad frente a outliers residuales.  


**4. Tipología de variables**

Se identifican cuatro grupos principales:

- Variables lineales y estables.

    - Características: baja asimetría, curtosis moderada, sin outliers relevantes, correlación útil con el target.
    - Ejemplos: amenity_count, attractions_within_radius, commercial_within_radius
    - Tratamiento: no requieren transformación, escalado estándar o MinMax.

- Variables con alta asimetría y curtosis.

    - Características: distribuciones sesgadas, colas largas, presencia de outliers, correlación significativa con el target.
    - Ejemplos: Variables de capacidad (accommodates, bathrooms, bedrooms, beds), Variables del host, minimum_nights.
    - Tratamiento: transformación logarítmica, escalado robusto, posible binning en casos discretos o extremos.

- Variables con señal débil o baja correlación.

    - Características: correlación baja con el target, distribuciones problemáticas (alta asimetría, curtosis extrema).
    - Ejemplos: host_total_listings_count (*corr ≈ 0.13*), review_scores_mean (*corr ≈ 0.13*), minimum_nights (*corr ≈ 0.03*).
    - Tratamiento: imputación por mediana, log-transform, binning para capturar patrones no lineales, validación de impacto en el modelo.

- Variables discretas o de baja varianza.

    - Características: valores concentrados en pocos niveles, poca dispersión, riesgo de baja señal si no se transforman.
    - Ejemplos: amenity_score, bathrooms (cuando toma valores enteros bajos), review_scores_mean (escala limitada 1–5)
    - Tratamiento: binning por categorías, escalado estándar o robusto según distribución.


**5. Estrategía de escalado**

El escalado constituye una etapa crítica en el preprocesamiento de variables numéricas, especialmente en modelos lineales y algoritmos sensibles a la magnitud de las características (ej. regresión, SVM, redes neuronales, KNN). Su objetivo es garantizar estabilidad numérica, mejorar la convergencia de los optimizadores y evitar que variables con rangos muy distintos dominen el aprendizaje.

La sugerencia del método de escalado en el diagnóstico se ha realizado de forma coherente con la naturaleza de cada variable:

- **MinMaxScaler** se asignó a variables acotadas o de conteo estable (`attractions_within_radius`, `commercial_within_radius`, `accommodates`, `bathrooms`, `bedrooms`), donde el rango es definido y los outliers no distorsionan significativamente. Este enfoque preserva la proporcionalidad y facilita la interpretación en modelos sensibles a rangos.

- **StandardScaler** se aplicó a variables con distribución aproximadamente normal y sin outliers relevantes (`amenity_count`, `amenity_score`). Al centrar en media cero y escalar a desviación estándar uno, se facilita la comparación entre variables y se estabilizan coeficientes en modelos lineales.

- **RobustScaler** se reservó para variables con alta asimetría, curtosis extrema y presencia de outliers (`dist_to_nearest_attraction`, `beds`, `host_total_listings_count`, `review_scores_mean`, `minimum_nights`). Este método utiliza la mediana y el rango intercuartílico, reduciendo la influencia de valores extremos y proporcionando mayor estabilidad.

**6. Consideraciones adicionales**

- **`review_scores_mean`:**  
  Aunque en el diagnóstico se sugiere la transformación logarítmica, esta variable está acotada en un rango limitado (1–5). Aplicar log no aporta beneficios significativos y puede distorsionar la escala. Es más adecuado tratarla como variable discreta o aplicar binning para capturar diferencias cualitativas en los niveles de puntuación.

- **`minimum_nights`:**  
  Presenta una distribución extremadamente sesgada y una correlación casi nula con el target. Más que una variable continua, su comportamiento es categórico: estancias cortas, medias y largas. Por ello, se recomienda segmentarla mediante **binning no lineal** (ej. [1–7], [8–30], [31+]) para reflejar mejor su impacto en el modelo.

- **`host_total_listings_count`:**  
  Exhibe señal débil y valores extremos que generan alta asimetría. Puede ser útil tras una transformación logarítmica combinada con binning, pero también debe evaluarse si su inclusión aporta valor real al modelo. En caso de no mejorar el desempeño, podría considerarse su reducción o exclusión para evitar ruido.

El diagnóstico no solo describe el comportamiento de las variables, sino que permite definir decisiones concretas de transformación. Esto marca la transición de un análisis exploratorio a un proceso de ingeniería de características orientado al modelado. Cada variable será en función de su comportamiento específico, evitando transformaciones arbitrarias y asegurando una preparación adecuada de los datos para etapas posteriores.

#### Comparación de transformaciones

In [171]:
# Display comparison tables for different transformations applied to each feature
print("\n======================== COMPARISON BETWEEN TRANSFORMATIONS FOR EACH FEATURE ========================\n")
compare_transformations(df_features, numeric_features)


======================== COMPARISON BETWEEN TRANSFORMATIONS FOR EACH FEATURE ========================


========== VARIABLE: dist_to_nearest_attraction ==========

    transformation    mean     std    skew  corr_with_target
0         original  1.3023  1.5888  2.9741           -0.2391
1              log  0.6883  0.4900  1.1976           -0.2894
2             sqrt  1.0009  0.5483  1.4068           -0.2808
3  winsor_moderate  1.2824  1.4795  2.3745           -0.2501
4    winsor_strong  1.2029  1.2059  1.6252           -0.2729

========== VARIABLE: attractions_within_radius ==========

    transformation    mean     std    skew  corr_with_target
0         original  2.7176  2.7997  0.6776            0.3042
1              log  0.9791  0.8540 -0.0071            0.3352
2             sqrt  1.2460  1.0794 -0.0012            0.3308
3  winsor_moderate  2.7170  2.7981  0.6745            0.3045
4    winsor_strong  2.6748  2.7040  0.5529            0.3160

========== VARIABLE: commercial_within_rad